# Simple Feature Exploration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from pathlib import Path
from skimage.color import rgb2gray
from skimage.filters import window, butterworth
from scipy.stats import gaussian_kde, wasserstein_distance
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, classification_report
from sklearn.ensemble import RandomForestClassifier
from scipy.ndimage import gaussian_filter1d
from matplotlib.colors import rgb_to_hsv

In [ ]:
metadata_full = pd.read_csv("./sample_images/metadata.csv", index_col=0)
cols_to_keep = ["image_id", "width", "height", "file_name", "common_name", "supercategory"]
metadata = metadata_full[cols_to_keep].copy()
metadata.head()

In [ ]:
plt.figure(figsize=(15,4))
metadata["supercategory"].value_counts().plot(kind='bar', edgecolor='black', rot=0)
plt.title("Supercategory distribution")
plt.xlabel("Category")
plt.ylabel("Count")
plt.show()

In [ ]:
sample_image_ids = metadata.groupby("supercategory")["image_id"].sample(n=1, random_state=1).tolist()
sample_df = metadata[metadata["image_id"].isin(sample_image_ids)]
sample_df

In [ ]:
data_dir = Path("./sample_images/images")
imgs = [np.array(Image.open(data_dir / path)) for path in sample_df["file_name"].tolist()]

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=3, figsize=(15,12))
axes = axs.flatten()
for i in range(len(imgs)):
    axes[i].imshow(imgs[i])
    axes[i].set_title(f"{sample_df["supercategory"].tolist()[i]}:\n{sample_df["common_name"].tolist()[i]}")

axes[-1].set_visible(False)

# Color Histogram

In [ ]:
def get_color_hist(img):
    colors = ('red', 'green', 'blue')
    result = {}
    for channel_id, color in enumerate(colors):
        hist, bin_edges = np.histogram(img[:, :, channel_id], bins=256, range=(0, 256))
        result[color]=hist
    result["bin_edges"]=bin_edges
    return result

In [ ]:
def collect_hists(metadata, categories):
    hists_by_cat = {}
    for cat in categories:
        imgs = metadata[metadata['supercategory']==cat]["image_id"].tolist()
        color_hists = {}
        for id in imgs:
            im_path = metadata[metadata["image_id"]==id]["file_name"].tolist()[0]
            img = np.array(Image.open(data_dir / im_path))
            single_hist = get_color_hist(img)

            if color_hists:
                color_hists['red'] + single_hist['red']
                color_hists['green'] + single_hist['green']
                color_hists['blue'] + single_hist['blue']
            else:
                color_hists = single_hist
        hists_by_cat[cat]=color_hists
    return hists_by_cat

In [ ]:
categories = set(metadata['supercategory'].tolist())
hists_by_cat = collect_hists(metadata, categories)

fig, axs = plt.subplots(ncols=4, nrows=3, figsize=(15,12))
axes = axs.flatten()
for i, cat in enumerate(categories):
    bin_edges = hists_by_cat[cat]["bin_edges"]
    for color in ["red", "green", "blue"]:
        hist = hists_by_cat[cat][color]
        axes[i].plot(bin_edges[:-1], hist, color=color)
    axes[i].set_xlim([0,256])
    axes[i].set_title(cat)

axes[-1].set_visible(False)
plt.show()

### Refining the features

Goal: one value per class (or a small feature vector)

Options:

- [NOPE] peak R, peak G, peak B - I can't guarantee that in a single image's histogram I will be able to find a peak
- [NOPE] area under each color curve - The area under each color curve is just the number of pixels that have any of that color (when measuring off the histogram)
- earth mover distance between each pair of colors, or between each color and the overall

In [ ]:
# earth mover distance
def wasserstein_features(hists):
    # input is hists where hists.keys = ["bin_edges", "red", "green", "blue"] for a single image
    edges = np.asarray(hists["bin_edges"], dtype=float)
    red = np.asarray(hists["red"], dtype=float)
    green = np.asarray(hists["green"], dtype=float)
    blue = np.asarray(hists["blue"], dtype=float)

    # use bin centers as the support points of each distribution
    centers = (edges[:-1] + edges[1:]) / 2.0

    # overall histogram: sum of the three channels per bin
    overall = red + green + blue

    # wasserstein distance between each channel and the overall histogram
    # (bin centers are the values, histogram counts are the weights)
    return [
        wasserstein_distance(centers, centers, u_weights=red, v_weights=overall),
        wasserstein_distance(centers, centers, u_weights=green, v_weights=overall),
        wasserstein_distance(centers, centers, u_weights=blue, v_weights=overall),
    ]

In [ ]:
def collect_wd(metadata, categories):
    wd_by_cat = {}
    for cat in categories:
        imgs = metadata[metadata['supercategory']==cat]["image_id"].tolist()
        wd = {'red': [], 'green': [], 'blue': []}
        for id in imgs:
            im_path = metadata[metadata["image_id"]==id]["file_name"].tolist()[0]
            img = np.array(Image.open(data_dir / im_path))
            single_wd = wasserstein_features(get_color_hist(img))

            wd['red'].append(single_wd[0])
            wd['green'].append(single_wd[1])
            wd['blue'].append(single_wd[2])
            
        wd_by_cat[cat]=wd
    return wd_by_cat

In [ ]:
categories = set(metadata['supercategory'].tolist())
wd_by_cat = collect_wd(metadata, categories)

fig, axs = plt.subplots(ncols=4, nrows=3, figsize=(15,12))
axes = axs.flatten()
for i, cat in enumerate(categories):
    for color in ["red", "green", "blue"]:
        wd = hists_by_cat[cat][color]
        hist, bin_edges = np.histogram(wd, bins=20)
        axes[i].plot(bin_edges[:-1], hist, color=color)
    axes[i].set_title(cat)

axes[-1].set_visible(False)
plt.show()

In [ ]:
def hist_features(hists, sigma=5):
    edges = np.asarray(hists["bin_edges"], dtype=float)
    centers = (edges[:-1] + edges[1:]) / 2.0
    feats, modes, means = [], [], []

    for color in ("red", "green", "blue"):
        h = np.asarray(hists[color], dtype=float)
        total = h.sum()

        # fraction of pixels in the saturated end bins
        feats += [h[-1] / total, h[0] / total]

        # drop the end bins for the shape statistics
        h_in, c_in = h[1:-1], centers[1:-1]
        p = h_in / max(h_in.sum(), 1e-12)

        mean = (p * c_in).sum()
        std = np.sqrt((p * (c_in - mean) ** 2).sum())
        skew = (p * ((c_in - mean) / (std + 1e-12)) ** 3).sum()
        kurt = (p * ((c_in - mean) / (std + 1e-12)) ** 4).sum()

        cdf = np.cumsum(p)
        q10, q50, q90 = np.interp([0.10, 0.50, 0.90], cdf, c_in)

        # smoothed mode: robust "peak" even when there isn't a strong one
        mode = c_in[np.argmax(gaussian_filter1d(h_in, sigma))]

        feats += [mean, std, skew, kurt, q10, q50, q90, mode]
        modes.append(mode)
        means.append(mean)

    # cross-channel relative features (R-G, R-B, G-B)
    for i, j in [(0, 1), (0, 2), (1, 2)]:
        feats += [modes[i] - modes[j], means[i] - means[j]]

    return np.array(feats)

In [ ]:
def hue_features(img, n_bins=18, sat_thresh=0.15, val_thresh=0.10):
    hsv = rgb_to_hsv(img[..., :3].astype(float) / 255.0)   # all channels in [0, 1]
    h, s, v = (hsv[..., k].ravel() for k in range(3))

    chromatic = (s > sat_thresh) & (v > val_thresh)
    feats = [chromatic.mean(), s.mean(), s.std(), v.mean(), v.std()]

    if chromatic.any():
        hc, w = h[chromatic], s[chromatic]          # weight hue by saturation
        angle = 2 * np.pi * hc
        C = np.sum(w * np.cos(angle)) / w.sum()
        S = np.sum(w * np.sin(angle)) / w.sum()
        R = np.hypot(C, S)                          # 0 = hues spread out, 1 = all one hue
        mean_angle = np.arctan2(S, C)
        hue_hist, _ = np.histogram(hc, bins=n_bins, range=(0, 1), weights=w)
        hue_hist = hue_hist / hue_hist.sum()
        feats += [np.cos(mean_angle), np.sin(mean_angle), R]
        feats += list(hue_hist)
    else:  # fully gray/white/black image
        feats += [0.0, 0.0, 0.0] + [0.0] * n_bins

    return np.array(feats)

In [ ]:
# prep data
data_dir = Path("./sample_images/images")
imgs = [np.array(Image.open(data_dir / path)) for path in metadata["file_name"].tolist()]
X = np.array([hue_features(img) for img in imgs]) # hue features
# X = np.array([hist_features(get_color_hist(img)) for img in imgs]) # histogram features
# X = np.array([wasserstein_features(get_color_hist(img)) for img in imgs]) # wasserstein version (doesn't perform well)
y = metadata["supercategory"].to_numpy()

# 60/20/20 stratified split
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=42
)

In [ ]:
# Logistic Regression

# def make_model_lr(C=1.0):
#     return make_pipeline(
#         StandardScaler(),
#         LogisticRegression(C=C, max_iter=1000),
#     )

# # Use the val set to choose C
# best_C, best_val_acc = None, -1
# for C in [0.01, 0.1, 1, 10, 100]:
#     model = make_model_lr(C).fit(X_train, y_train)
#     val_acc = accuracy_score(y_val, model.predict(X_val))
#     print(f"C={C:<6} val accuracy = {val_acc:.3f}")
#     if val_acc > best_val_acc:
#         best_C, best_val_acc = C, val_acc
# print(f"Best C: {best_C}")

# # Refit on train+val, evaluate once on test
# final_model = make_model_lr(best_C).fit(X_trainval, y_trainval)
# y_pred = final_model.predict(X_test)

In [ ]:
# Random Forest

def make_model(max_depth=None):
    return RandomForestClassifier(
        n_estimators=300,
        max_depth=max_depth,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42,
    )

# Use the val set to choose max_depth
best_param, best_val_acc = None, -1
for max_depth in [3, 5, 10, 20, None]:
    model = make_model(max_depth).fit(X_train, y_train)
    val_acc = accuracy_score(y_val, model.predict(X_val))
    print(f"max_depth={str(max_depth):<5} val accuracy = {val_acc:.3f}")
    if val_acc > best_val_acc:
        best_param, best_val_acc = max_depth, val_acc
print(f"Best max_depth: {best_param}")

# Refit on train+val, evaluate on test
final_model = make_model(best_param).fit(X_trainval, y_trainval)
y_pred = final_model.predict(X_test)

In [ ]:
# majority-class baseline for context
baseline = DummyClassifier(strategy="most_frequent").fit(X_trainval, y_trainval)
print(f"\nBaseline (majority class) test accuracy: {accuracy_score(y_test, baseline.predict(X_test)):.3f}")
print(f"Logistic regression test accuracy:       {accuracy_score(y_test, y_pred):.3f}\n")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, xticks_rotation=45, cmap="Blues", ax=ax
)
ax.set_title("Test set confusion matrix")
plt.tight_layout()
plt.show()

In [ ]:
print(final_model.feature_importances_)
top5_indices = np.argpartition(final_model.feature_importances_, -5)[-5:]
print(top5_indices)

# RGB
# 26 = blue q10
# 21 = blue frac_0
# 35 = mean_G-B
# 33 = mean_R-B
# 31 = mean_R-G

# HSV
# 6 = sine of mean hue angle
# 2 = saturation std
# 5 = cosine of mean hue angle
# 12 = yellow-green
# 9 = orange

# Band Pass Ratio Filter (fine vs coarse)

In [ ]:
def get_BPR(img, cutoff=0.25, order=2):
    """
    Ratio of fine (high-pass) to coarse (low-pass) energy.

    cutoff : split frequency as a fraction of Nyquist (0 < cutoff <= 1).
    order  : Butterworth filter order (higher = sharper transition).
    """
    gray = rgb2gray(img)
    gray = (gray - gray.mean()) * window('hann', gray.shape)

    fc = cutoff * 0.5
    coarse = butterworth(gray, fc, high_pass=False, order=order)
    fine = butterworth(gray, fc, high_pass=True, order=order)

    return np.sum(fine ** 2) / (np.sum(coarse ** 2) + 1e-12)

In [ ]:
def collect_BPR(metadata, categories):
    ratios_by_cat = {}
    for cat in categories:
        imgs = metadata[metadata['supercategory']==cat]["image_id"].tolist()
        img_ratios = []
        for id in imgs:
            im_path = metadata[metadata["image_id"]==id]["file_name"].tolist()[0]
            img = np.array(Image.open(data_dir / im_path))
            bpr = get_BPR(img)

            img_ratios.append(bpr)
        ratios_by_cat[cat]=img_ratios
    return ratios_by_cat

In [ ]:
categories = set(metadata['supercategory'].tolist())
ratios_by_cat = collect_BPR(metadata, categories)

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=3, figsize=(15,12), sharex=True, sharey=True)
axes = axs.flatten()
for i, cat in enumerate(categories):
    data = ratios_by_cat[cat]
    kde = gaussian_kde(data)
    x_vals = np.linspace(min(data), max(data), 200)
    y_vals = kde(x_vals)

    axes[i].plot(x_vals, y_vals, color='blue', linewidth=2)
    axes[i].fill_between(x_vals, y_vals, color='blue', alpha=0.3)

    axes[i].set_title(cat)
    axes[i].tick_params(axis='x', labelbottom=True)

axes[-1].set_visible(False)
plt.suptitle("Ratio of high pass energy to low pass energy", y=0.92)
plt.show()

The peaks for this look very similar in position, so I don't think it would make a good feature, despite the distribution of them being different between classes.

# HOG (Histogram of Oriented Gradients)

computes horizontal and vertical intensity changes for every pixel to find edge magnitude and angle, builds a histogram of gradient angles weighted by magnitude, then flattens it.


I want to build the feature that would be used for classifying, somehow visualize the total or average HOG for each class so I can evaluate if it will be an effective feature for classification. Then I want to extract some properties of the HOG (e.g. a measure of the histogram's shape, or specific bin values) and look at the distribution of those features for each class. Lastly, I want to use them to train either a logistic regression or random forest model (using existing code) and visualize the results. using skimage.feature.hog function, and its visualize=True option.

In [ ]:
import numpy as np
from skimage.color import rgb2gray
from skimage.feature import hog
from skimage.transform import resize

IMG_SIZE = (128, 128)   # fixed size, so every image gives a same-length vector
N_ORIENT = 9            # unsigned gradients: bins span 0-180 degrees


# def _to_gray(img):
#     img = np.asarray(img)
#     if img.ndim == 2:
#         g = img.astype(float) / 255.0
#     else:
#         g = rgb2gray(img[..., :3].astype(float) / 255.0)   # drops alpha if present
#     return resize(g, IMG_SIZE, anti_aliasing=True)

def _trim_bars(g, tol=0.02):
    """Drop leading/trailing rows and columns that are uniform near-black or near-white."""
    def keep(stats_std, stats_mean):
        bad = (stats_std < tol) & ((stats_mean < 0.05) | (stats_mean > 0.95))
        good = np.flatnonzero(~bad)
        return (good[0], good[-1] + 1) if len(good) else (0, len(bad))
    r0, r1 = keep(g.std(axis=1), g.mean(axis=1))
    c0, c1 = keep(g.std(axis=0), g.mean(axis=0))
    return g[r0:r1, c0:c1]

def _to_gray(img, crop=0.08):
    g = _trim_bars(_gray_native(img))
    h, w = g.shape
    dh, dw = int(h * crop), int(w * crop)
    g = g[dh:h - dh, dw:w - dw]
    return resize(g, IMG_SIZE, anti_aliasing=True)


def hog_descriptor(img, orientations=N_ORIENT, pixels_per_cell=(16, 16), cells_per_block=(2, 2)):
    """Full flattened HOG vector (fixed length). Use this for the per-class
    average/total HOG visualization, e.g. np.mean(descs[y == c], axis=0)."""
    return hog(
        _to_gray(img),
        orientations=orientations,
        pixels_per_cell=pixels_per_cell,
        cells_per_block=cells_per_block,
        block_norm="L2-Hys",
        feature_vector=True,
    )


def _refined_peaks(p):
    """Sub-bin angles (radians, in [0, pi)) of circular local maxima of p, strongest first."""
    n = len(p)
    is_peak = (p > np.roll(p, 1)) & (p >= np.roll(p, -1))
    ks = np.flatnonzero(is_peak)
    ks = ks[np.argsort(p[ks])[::-1]]
    angles = []
    for k in ks:
        y0, y1, y2 = p[k - 1], p[k], p[(k + 1) % n]
        off = 0.5 * (y0 - y2) / (y0 - 2 * y1 + y2)      # denominator < 0 at a peak
        angles.append(((k + 0.5 + off) * np.pi / n) % np.pi)
    return angles


def hog_features(img, orientations=N_ORIENT, pixels_per_cell=(16, 16), cells_per_block=(2, 2)):
    gray = _to_gray(img)

    blocks = hog(
        gray,
        orientations=orientations,
        pixels_per_cell=pixels_per_cell,
        cells_per_block=cells_per_block,
        block_norm="L2-Hys",
        feature_vector=False,
    )

    ori_hist = blocks.reshape(-1, orientations).sum(axis=0)
    total = ori_hist.sum()
    if total <= 0:
        return np.zeros(len(hog_feature_names(orientations)))
    p = ori_hist / total

    # --- existing shape features ---
    entropy = -np.sum(p[p > 0] * np.log(p[p > 0])) / np.log(orientations)
    peak = p.max()
    theta = (np.arange(orientations) + 0.5) * np.pi / orientations
    C = np.sum(p * np.cos(2 * theta))
    S = np.sum(p * np.sin(2 * theta))
    R = np.hypot(C, S)
    mean_angle = np.arctan2(S, C)

    # --- NEW 1: angle between the two strongest orientation peaks (degrees, 0-90) ---
    # 0 means only one peak exists; real gaps are never exactly 0, so it's a clean sentinel.
    a = _refined_peaks(p)
    if len(a) >= 2:
        d = abs(a[0] - a[1])
        peak_gap = np.degrees(min(d, np.pi - d))
    else:
        peak_gap = 0.0

    # --- gradient magnitude stats ---
    gy, gx = np.gradient(gray)
    mag = np.hypot(gx, gy)
    h, w = mag.shape
    center = mag[h // 4: 3 * h // 4, w // 4: 3 * w // 4]
    center_frac = center.sum() / (mag.sum() + 1e-12)
    edge_density = (mag > 0.05).mean()

    # --- NEW 2 & 3: PCA of the spatial distribution of gradient energy ---
    yy, xx = np.mgrid[0:h, 0:w]
    xx = xx / (w - 1)
    yy = yy / (h - 1)                              # unit-square coordinates
    wgt = mag / (mag.sum() + 1e-12)
    mx, my = (wgt * xx).sum(), (wgt * yy).sum()
    cov = np.array([
        [(wgt * (xx - mx) ** 2).sum(),            (wgt * (xx - mx) * (yy - my)).sum()],
        [(wgt * (xx - mx) * (yy - my)).sum(),     (wgt * (yy - my) ** 2).sum()],
    ])
    evals, evecs = np.linalg.eigh(cov)             # ascending
    lam2, lam1 = evals
    phi = np.arctan2(evecs[1, 1], evecs[0, 1])     # major axis; sign of eigenvector is arbitrary,
    pc_cos, pc_sin = np.cos(2 * phi), np.sin(2 * phi)  # so encode mod 180 via the doubled angle
    # pc_cos = +1: edge mass is spread horizontally (wide); -1: vertically (tall)

    feats = [mag.mean(), mag.std(), edge_density, center_frac,
             entropy, peak, np.cos(mean_angle), np.sin(mean_angle), R,
             peak_gap, pc_cos, pc_sin, lam1, lam2]
    feats += list(p)
    return np.array(feats)


def hog_feature_names(orientations=N_ORIENT):
    return (["mag_mean", "mag_std", "edge_density", "center_frac",
             "ori_entropy", "ori_peak", "ori_cos", "ori_sin", "ori_R",
             "peak_gap_deg", "pc_cos", "pc_sin", "pc_eig1", "pc_eig2"]
            + [f"ori_bin_{k}" for k in range(orientations)])

In [ ]:
# prep data
data_dir = Path("./sample_images/images")
imgs = [np.array(Image.open(data_dir / path)) for path in metadata["file_name"].tolist()]
X = np.array([hog_features(img) for img in imgs]) # HOG features
# X = np.array([hue_features(img) for img in imgs]) # hue features
# X = np.array([hist_features(get_color_hist(img)) for img in imgs]) # histogram features
# X = np.array([wasserstein_features(get_color_hist(img)) for img in imgs]) # wasserstein version (doesn't perform well)
y = metadata["supercategory"].to_numpy()

# 60/20/20 stratified split
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=42
)

In [ ]:
# # Logistic Regression

# def make_model_lr(C=1.0):
#     return make_pipeline(
#         StandardScaler(),
#         LogisticRegression(C=C, max_iter=1000),
#     )

# # Use the val set to choose C
# best_C, best_val_acc = None, -1
# for C in [0.01, 0.1, 1, 10, 100]:
#     model = make_model_lr(C).fit(X_train, y_train)
#     val_acc = accuracy_score(y_val, model.predict(X_val))
#     print(f"C={C:<6} val accuracy = {val_acc:.3f}")
#     if val_acc > best_val_acc:
#         best_C, best_val_acc = C, val_acc
# print(f"Best C: {best_C}")

# # Refit on train+val, evaluate once on test
# final_model = make_model_lr(best_C).fit(X_trainval, y_trainval)
# y_pred = final_model.predict(X_test)

In [ ]:
# Random Forest

def make_model(max_depth=None):
    return RandomForestClassifier(
        n_estimators=300,
        max_depth=max_depth,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42,
    )

# Use the val set to choose max_depth
best_param, best_val_acc = None, -1
for max_depth in [3, 5, 10, 20, None]:
    model = make_model(max_depth).fit(X_train, y_train)
    val_acc = accuracy_score(y_val, model.predict(X_val))
    print(f"max_depth={str(max_depth):<5} val accuracy = {val_acc:.3f}")
    if val_acc > best_val_acc:
        best_param, best_val_acc = max_depth, val_acc
print(f"Best max_depth: {best_param}")

# Refit on train+val, evaluate on test
final_model = make_model(best_param).fit(X_trainval, y_trainval)
y_pred = final_model.predict(X_test)

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, xticks_rotation=45, cmap="Blues", ax=ax
)
ax.set_title("Test set confusion matrix")
plt.tight_layout()
plt.show()

In [ ]:
# readable importances
names = hog_feature_names()
top5_indices = np.argpartition(final_model.feature_importances_, -5)[-5:]
for i in top5_indices[np.argsort(final_model.feature_importances_[top5_indices])[::-1]]:
    print(names[i], final_model.feature_importances_[i])

# mag_std is high when the image has a sharp sugject on a smooth background
# center_frac is high when the total gradient energy is concentrated in the center of the image
# edge_density is high when there are a lot of edges in the image
# mag_mean is high when the average edge is very sharp (high contrast)
# pc_eig1 is high when the principal direction carries a lot of info about the distribution of angles (?)

In [ ]:
def orientation_hists(D, n_orient=N_ORIENT):
    """(n_images, descriptor_len) -> (n_images, n_orient), each row sums to 1."""
    h = D.reshape(len(D), -1, n_orient).sum(axis=1)   # sum over all blocks/cells
    return h / h.sum(axis=1, keepdims=True)

def plot_class_hog(D, y, n_orient=N_ORIENT, subtract_global=False, ncols=4):
    H = orientation_hists(D, n_orient)
    global_mean = H.mean(axis=0)
    centers = (np.arange(n_orient) + 0.5) * 180 / n_orient    # degrees

    categories = sorted(set(y))
    nrows = int(np.ceil(len(categories) / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(15, 3.5 * nrows), sharey=True)
    axes = np.atleast_1d(axs).flatten()

    for ax, cat in zip(axes, categories):
        Hc = H[y == cat]
        mean, std = Hc.mean(axis=0), Hc.std(axis=0)
        if subtract_global:
            mean = mean - global_mean
        ax.plot(centers, mean, color="k", marker="o")
        ax.fill_between(centers, mean - std, mean + std, color="k", alpha=0.15)
        if subtract_global:
            ax.axhline(0, color="gray", lw=0.8)
        ax.set_title(f"{cat} (n={len(Hc)})")
        ax.set_xticks([0, 45, 90, 135, 180])
        ax.set_xlabel("gradient orientation (deg)")

    for ax in axes[len(categories):]:
        ax.set_visible(False)
    axes[0].set_ylabel("deviation from global mean" if subtract_global else "fraction of HOG energy")
    plt.tight_layout()
    plt.show()

D = np.array([hog_descriptor(img) for img in imgs])
y = metadata["supercategory"].to_numpy()
plot_class_hog(D, y)                          # absolute per-class orientation histogram
plot_class_hog(D, y, subtract_global=True)    # what makes each class different

In [ ]:
from matplotlib.collections import LineCollection

def cell_hists(D, pixels_per_cell=(16, 16), cells_per_block=(2, 2), n_orient=N_ORIENT):
    """Undo the block structure: (n_images, desc_len) -> (n_images, n_cell_rows, n_cell_cols, n_orient).
    Overlapping blocks each hold a normalised copy of a cell; average the copies."""
    ncr, ncc = IMG_SIZE[0] // pixels_per_cell[0], IMG_SIZE[1] // pixels_per_cell[1]
    cbr, cbc = cells_per_block
    nbr, nbc = ncr - cbr + 1, ncc - cbc + 1
    B = D.reshape(len(D), nbr, nbc, cbr, cbc, n_orient)
    out = np.zeros((len(D), ncr, ncc, n_orient))
    cnt = np.zeros((ncr, ncc, 1))
    for di in range(cbr):
        for dj in range(cbc):
            out[:, di:di + nbr, dj:dj + nbc] += B[:, :, :, di, dj]
            cnt[di:di + nbr, dj:dj + nbc] += 1
    return out / cnt


def plot_class_hog_glyphs(D, y, n_orient=N_ORIENT, ncols=4, **hog_kwargs):
    """Average HOG per class, drawn as oriented line segments (edge direction,
    i.e. perpendicular to the gradient) with length proportional to strength."""
    C = cell_hists(D, n_orient=n_orient, **hog_kwargs)
    categories = sorted(set(y))
    means = {cat: C[y == cat].mean(axis=0) for cat in categories}
    vmax = max(m.max() for m in means.values())            # shared scale -> comparable across classes
    ncr, ncc = C.shape[1:3]
    theta = (np.arange(n_orient) + 0.5) * np.pi / n_orient  # gradient angle per bin
    phi = theta + np.pi / 2                                 # edge direction

    nrows = int(np.ceil(len(categories) / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = np.atleast_1d(axs).flatten()
    for ax, cat in zip(axes, categories):
        m = means[cat]
        segs = []
        widths = []
        for r in range(ncr):
            for c in range(ncc):
                cx, cy = c + 0.5, r + 0.5
                for k in range(n_orient):
                    half = 0.5 * m[r, c, k] / vmax
                    dx, dy = half * np.cos(phi[k]), half * np.sin(phi[k])
                    segs.append([(cx - dx, cy - dy), (cx + dx, cy + dy)])
        ax.add_collection(LineCollection(segs, colors="white", linewidths=1.2))
        ax.set_xlim(0, ncc); ax.set_ylim(ncr, 0)            # image coordinates (y down)
        ax.set_aspect("equal"); ax.set_facecolor("black")
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"{cat} (n={(y == cat).sum()})")
    for ax in axes[len(categories):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()


def plot_class_anisotropy_maps(D, y, n_orient=N_ORIENT, ncols=4, **hog_kwargs):
    """Per-cell signed edge-orientation balance, averaged per class.
    +1 = gradients horizontal (vertical edges), -1 = gradients vertical (horizontal edges)."""
    C = cell_hists(D, n_orient=n_orient, **hog_kwargs)
    theta = (np.arange(n_orient) + 0.5) * np.pi / n_orient
    w = np.cos(2 * theta)
    categories = sorted(set(y))
    maps = {}
    for cat in categories:
        m = C[y == cat].mean(axis=0)
        maps[cat] = (m * w).sum(-1) / (m.sum(-1) + 1e-12)
    lim = max(np.abs(a).max() for a in maps.values())

    nrows = int(np.ceil(len(categories) / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.6 * nrows))
    axes = np.atleast_1d(axs).flatten()
    for ax, cat in zip(axes, categories):
        im = ax.imshow(maps[cat], cmap="RdBu_r", vmin=-lim, vmax=lim)
        ax.set_title(cat); ax.set_xticks([]); ax.set_yticks([])
    for ax in axes[len(categories):]:
        ax.set_visible(False)
    fig.colorbar(im, ax=axes.tolist(), shrink=0.6,
                 label="vertical edges (+)  <->  horizontal edges (-)")
    plt.show()


plot_class_hog_glyphs(D, y)
plot_class_anisotropy_maps(D, y)

In [ ]:
def _gray_native(img):
    img = np.asarray(img)
    return (img.astype(float) / 255.0) if img.ndim == 2 else rgb2gray(img[..., :3].astype(float) / 255.0)

def border_stats(imgs, k=4, tol=0.02):
    """Fraction of images whose outer k-pixel strip is uniform AND near-black/white (i.e. a bar)."""
    hits = dict(top=0, bottom=0, left=0, right=0)
    for img in imgs:
        g = _gray_native(img)
        strips = dict(top=g[:k], bottom=g[-k:], left=g[:, :k], right=g[:, -k:])
        for name, a in strips.items():
            if a.std() < tol and (a.mean() < 0.05 or a.mean() > 0.95):
                hits[name] += 1
    return {name: v / len(imgs) for name, v in hits.items()}

print(border_stats(imgs))

# How deep does the effect go? Anisotropy by distance from the edge, all classes pooled.
C = cell_hists(D)
m = C.mean(axis=0)
theta = (np.arange(N_ORIENT) + 0.5) * np.pi / N_ORIENT
A = (m * np.cos(2 * theta)).sum(-1) / m.sum(-1)
print("by row (top -> bottom):   ", np.round(A[:, 1:-1].mean(axis=1), 3))
print("by col (left -> right):   ", np.round(A[1:-1, :].mean(axis=0), 3))

# Eigenvector based?

PCA on the image?

PCA on the HOG?

